In [1]:
import pandas as pd

data = pd.read_csv('../eeg-analysis/go nogo/go nogo/Go NoGo_afif.txt', sep=',', skiprows=4)

In [2]:
import os
from glob import glob

go_nogo_dir = r'../eeg-analysis/go nogo/go nogo'
resting_dir = r'../eeg-analysis/resting/resting'

go_nogo_files = sorted(glob(os.path.join(go_nogo_dir, '*.txt')))
resting_files = sorted(glob(os.path.join(resting_dir, '*.txt')))
all_files = go_nogo_files + resting_files

print(f'Go-NoGo files: {len(go_nogo_files)}')
print(f'Resting files: {len(resting_files)}')
print(f'Total files: {len(all_files)}')

all_data = []
for fp in all_files:
    df_tmp = pd.read_csv(fp, sep=',', skiprows=4)
    all_data.append(df_tmp)

# Data gabungan dari semua subjek (go nogo + resting)
data = pd.concat(all_data, ignore_index=True)
print('Combined raw shape:', data.shape)

Go-NoGo files: 15
Resting files: 10
Total files: 25
Combined raw shape: (1594633, 33)


In [3]:
eeg_channels = [f' EXG Channel {i}' for i in range(16)]
eeg_data = data[eeg_channels].values.T  # (16, n_samples)

In [4]:
import numpy as np
from scipy.signal import resample, iirnotch, filtfilt, butter, welch

def upsample_signal(signal, fs_old=125, fs_new=512):
    n_samples = int(len(signal) * fs_new / fs_old)
    return resample(signal, n_samples)

def notch_filter(signal, fs, freq=50, Q=30):
    b, a = iirnotch(freq, Q, fs)
    return filtfilt(b, a, signal)

def bandpass_filter(signal, fs, low=0.5, high=40, order=4):
    nyquist = 0.5 * fs
    low /= nyquist
    high /= nyquist
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal)

def scaling(signal):
    return (signal - np.mean(signal)) / np.std(signal)

def compute_psd(signal, fs):
    freqs, psd = welch(signal, fs=fs, nperseg=256)
    return psd

def extract_feature(psd):
    return np.mean(psd)

In [5]:
processed_channels = []

for ch in eeg_data:
    signal = upsample_signal(ch, 125, 512)
    signal = notch_filter(signal, 512)
    signal = bandpass_filter(signal, 512)
    processed_channels.append(signal)

processed_channels = np.array(processed_channels)  
# shape: (16, n_samples_baru)

In [6]:
window_size = 2 * 512  # 1024
step = window_size

n_samples = processed_channels.shape[1]

dataset_features = []

for start in range(0, n_samples - window_size, step):
    window_features = []
    
    for ch in processed_channels:
        segment = ch[start:start + window_size]
        
        # scaling per window
        segment = scaling(segment)
        
        # PSD
        psd = compute_psd(segment, 512)
        
        # mean PSD
        feat = extract_feature(psd)
        window_features.append(feat)
    
    dataset_features.append(window_features)

dataset_features = np.array(dataset_features)

In [7]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 1))

dataset_features_scaled = scaler.fit_transform(dataset_features)

In [8]:
import pandas as pd

df = pd.DataFrame(dataset_features_scaled, 
                  columns=[f'Ch_{i}' for i in range(16)])

df.to_csv('eeg_features_windowed_scaled.csv', index=False)

GABUNG DATASET

In [9]:
import pandas as pd

# Paths
cognitive_path = r'../eeg-analysis/cognitive_state_discrimination_dataset - cognitive_state_discrimination_dataset.csv'
feature_path = r'eeg_features_windowed_scaled.csv'
output_path = r'../eeg-analysis/noncognitive.csv'

# Channel urutan wajib (sesuai permintaan)
selected_channels = [1, 2, 5, 6, 10, 11, 23, 24, 27, 28, 8, 9, 16, 17, 21, 22]
feature_cols = [f'Channel_{ch}_PSD' for ch in selected_channels]

# Load datasets
cog = pd.read_csv(cognitive_path)
feat = pd.read_csv(feature_path)

# Ambil hanya channel terpilih dari cognitive + label
cog_selected = cog[feature_cols + ['Target_Label']].copy()

# Gunakan label 0/1/2 dari cognitive sebagai basis (label 3 akan diisi dari feature_path)
cog_base = cog_selected[cog_selected['Target_Label'] != 3].copy()

# Hitung jumlah data per label di cognitive untuk acuan jumlah label 3
label_counts = cog_base['Target_Label'].value_counts().sort_index()
print('Cognitive label counts (without label 3):')
print(label_counts)

# Pilih salah satu label acuan (default: label 0 jika ada, jika tidak ambil label pertama)
if 0 in label_counts.index:
    ref_label = 0
else:
    ref_label = label_counts.index[0]
target_count = int(label_counts.loc[ref_label])
print(f'Using label {ref_label} count as target for label 3: {target_count}')

# Ubah kolom Ch_0..Ch_15 -> Channel sesuai urutan yang sama
mapping = {f'Ch_{i}': f'Channel_{ch}_PSD' for i, ch in enumerate(selected_channels)}
feat_selected = feat.rename(columns=mapping).copy()
feat_selected = feat_selected[feature_cols].copy()

# Samakan jumlah data label 3 agar mirip dengan salah satu label cognitive
replace_needed = len(feat_selected) < target_count
feat_selected = feat_selected.sample(n=target_count, random_state=42, replace=replace_needed).copy()
feat_selected['Target_Label'] = 3

# Gabungkan
merged = pd.concat([cog_base, feat_selected], ignore_index=True)

# Header tetap berurutan 1..16 + label
sorted_channels = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
new_header = [f'Channel_{ch}_PSD' for ch in sorted_channels] + ['Target_Label']
merged.columns = new_header

# Simpan
merged.to_csv(output_path, index=False)

print('Saved:', output_path)
print('Shape:', merged.shape)
print('Columns:', merged.columns.tolist())
print('Label counts after merge:')
print(merged['Target_Label'].value_counts().sort_index())

Cognitive label counts (without label 3):
Target_Label
0    10
1    10
2    10
Name: count, dtype: int64
Using label 0 count as target for label 3: 10
Saved: ../eeg-analysis/noncognitive.csv
Shape: (40, 17)
Columns: ['Channel_1_PSD', 'Channel_2_PSD', 'Channel_3_PSD', 'Channel_4_PSD', 'Channel_5_PSD', 'Channel_6_PSD', 'Channel_7_PSD', 'Channel_8_PSD', 'Channel_9_PSD', 'Channel_10_PSD', 'Channel_11_PSD', 'Channel_12_PSD', 'Channel_13_PSD', 'Channel_14_PSD', 'Channel_15_PSD', 'Channel_16_PSD', 'Target_Label']
Label counts after merge:
Target_Label
0    10
1    10
2    10
3    10
Name: count, dtype: int64
